# 05. 규제 이벤트 원자화

**목적**: `regulation_events(2015~).csv`를 API 호출 단위로 원자화

**처리 내용**:
1. 세미콜론으로 연결된 HS 코드를 행 단위로 분리
2. 수집 기간(`window_start`, `window_end`) 계산: `start_date ± 12개월`

**출력**: `data/interim/regulation_events_atomic.csv`

In [1]:
from pathlib import Path

import pandas as pd
from dateutil.relativedelta import relativedelta

In [9]:
DATA_DIR = Path("../data/interim")
INPUT_PATH = DATA_DIR / "regulation_events(2015~).csv"
OUTPUT_PATH = DATA_DIR / "regulation_events_atomic(~2015).csv"

WINDOW_MONTHS = 12

## 1. 데이터 로드 & 기본 검사

In [3]:
df = pd.read_csv(INPUT_PATH)

print("Shape:", df.shape)
print("\nNull counts:")
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

Shape: (104, 18)

Null counts:
event_id                    0
source_row_id               0
origin_country_name_kr      0
origin_country_iso3         0
product_name_kr             0
product_name_normalized     0
hs_code                     0
duty_text_raw               0
duty_type                   0
duty_rate_min               1
duty_rate_max               1
has_price_undertaking       0
has_reference_price_diff    0
has_partial_exclusion       0
start_date                  0
end_date                    0
duration_days               0
is_active_as_of_extract     0
dtype: int64

Duplicates: 0


In [4]:
df.head(3)

,event_id,source_row_id,origin_country_name_kr,origin_country_iso3,product_name_kr,product_name_normalized,hs_code,duty_text_raw,duty_type,duty_rate_min,duty_rate_max,has_price_undertaking,has_reference_price_diff,has_partial_exclusion,start_date,end_date,duration_days,is_active_as_of_extract
0,AD-01-01,1,중국,CHN,플로트판유리(2차재심),플로트판유리,700529,"12.04~36.01,가격약속",mixed,12.04,36.01,True,False,False,2015-01-07,2018-01-06,1096,False
1,AD-02-01,2,중국,CHN,도자기질 타일(2차재심),도자기질 타일,690721;690722;690723,9.07~29.41,ad_valorem,9.07,29.41,False,False,False,2015-02-25,2018-02-24,1096,False
2,AD-03-01,3,중국,CHN,H형강,H형강,721633,"28.23~32.71,가격약속",mixed,28.23,32.71,True,False,False,2015-07-30,2020-07-29,1827,False


## 2. HS 코드 세미콜론 분리

In [5]:
# 세미콜론 포함 행 확인
multi_hs = df[df["hs_code"].str.contains(";", na=False)]
print(f"세미콜론 포함 행: {len(multi_hs)}건")
multi_hs[["event_id", "product_name_kr", "hs_code"]]

세미콜론 포함 행: 28건


,event_id,product_name_kr,hs_code
1,AD-02-01,도자기질 타일(2차재심),690721;690722;690723
11,AD-09-01,스테인리스스틸 후판(1차재심),721921;721922;721923
16,AD-13-01,스테인레스스틸바(3차재심),722211;722219;722220
17,AD-13-02,스테인레스스틸바(3차재심),722211;722219;722220
18,AD-13-03,스테인레스스틸바(3차재심),722211;722219;722220
26,AD-17-01,도공인쇄용지,481013;481019
27,AD-17-02,도공인쇄용지,481013;481019
28,AD-17-03,도공인쇄용지,481013;481019
30,AD-19-01,에탄올아민(1차재심),292211;292212;292213
31,AD-19-02,에탄올아민(1차재심),292211;292212;292213


In [5]:
def explode_hs_codes(df: pd.DataFrame) -> pd.DataFrame:
    """세미콜론으로 연결된 hs_code를 행 단위로 분리."""
    df_clean = df.copy()
    df_clean["hs_code"] = df_clean["hs_code"].str.split(";")
    df_clean = df_clean.explode("hs_code").reset_index(drop=True)
    df_clean["hs_code"] = df_clean["hs_code"].str.strip()
    return df_clean


df_exploded = explode_hs_codes(df)
print(f"원본: {len(df)}행 → 원자화 후: {len(df_exploded)}행")

원본: 104행 → 원자화 후: 155행


## 3. 수집 기간(window) 계산

In [6]:
def add_collection_window(df: pd.DataFrame, months: int = 12) -> pd.DataFrame:
    """start_date 기준 ±months 수집 기간 컬럼 추가."""
    df_clean = df.copy()
    df_clean["start_date"] = pd.to_datetime(df_clean["start_date"])
    df_clean["window_start"] = df_clean["start_date"].apply(
        lambda d: (d - relativedelta(months=months)).strftime("%Y-%m")
    )
    df_clean["window_end"] = df_clean["start_date"].apply(
        lambda d: (d + relativedelta(months=months)).strftime("%Y-%m")
    )
    return df_clean


df_atomic = add_collection_window(df_exploded, months=WINDOW_MONTHS)

# 결과 확인
df_atomic[["event_id", "product_name_kr", "hs_code", "start_date", "window_start", "window_end"]].head(10)

,event_id,product_name_kr,hs_code,start_date,window_start,window_end
0,AD-01-01,플로트판유리(2차재심),700529,2015-01-07,2014-01,2016-01
1,AD-02-01,도자기질 타일(2차재심),690721,2015-02-25,2014-02,2016-02
2,AD-02-01,도자기질 타일(2차재심),690722,2015-02-25,2014-02,2016-02
3,AD-02-01,도자기질 타일(2차재심),690723,2015-02-25,2014-02,2016-02
4,AD-03-01,H형강,721633,2015-07-30,2014-07,2016-07
5,AD-04-01,공기압 전송용 밸브,848120,2015-08-19,2014-08,2016-08
6,AD-05-01,초산에틸(2차재심),291531,2015-11-19,2014-11,2016-11
7,AD-05-02,초산에틸(2차재심),291531,2015-11-19,2014-11,2016-11
8,AD-05-03,초산에틸(2차재심),291531,2015-11-19,2014-11,2016-11
9,AD-06-01,초산에틸,291531,2015-11-19,2014-11,2016-11


In [8]:
# 세미콜론 분리 전후 검증
sample_event = "AD-02-01"
print(f"[{sample_event}] 원자화 결과:")
df_atomic[df_atomic["event_id"] == sample_event][
    ["event_id", "product_name_kr", "hs_code", "window_start", "window_end"]
]

[AD-02-01] 원자화 결과:


,event_id,product_name_kr,hs_code,window_start,window_end
1,AD-02-01,도자기질 타일(2차재심),690721,2014-02,2016-02
2,AD-02-01,도자기질 타일(2차재심),690722,2014-02,2016-02
3,AD-02-01,도자기질 타일(2차재심),690723,2014-02,2016-02


## 4. 최종 확인 및 저장

In [7]:
print("Shape:", df_atomic.shape)
print("\nColumns:", df_atomic.columns.tolist())
print("\nNull counts:")
print(df_atomic.isnull().sum())
print("\nwindow_start 샘플:", df_atomic["window_start"].head(3).tolist())
print("window_end 샘플:  ", df_atomic["window_end"].head(3).tolist())

Shape: (155, 20)

Columns: ['event_id', 'source_row_id', 'origin_country_name_kr', 'origin_country_iso3', 'product_name_kr', 'product_name_normalized', 'hs_code', 'duty_text_raw', 'duty_type', 'duty_rate_min', 'duty_rate_max', 'has_price_undertaking', 'has_reference_price_diff', 'has_partial_exclusion', 'start_date', 'end_date', 'duration_days', 'is_active_as_of_extract', 'window_start', 'window_end']

Null counts:
event_id                    0
source_row_id               0
origin_country_name_kr      0
origin_country_iso3         0
product_name_kr             0
product_name_normalized     0
hs_code                     0
duty_text_raw               0
duty_type                   0
duty_rate_min               3
duty_rate_max               3
has_price_undertaking       0
has_reference_price_diff    0
has_partial_exclusion       0
start_date                  0
end_date                    0
duration_days               0
is_active_as_of_extract     0
window_start                0
window_end 

In [10]:
df_atomic.to_csv(OUTPUT_PATH, index=False)
print(f"저장 완료: {OUTPUT_PATH}")
print(f"총 {len(df_atomic)}행")

저장 완료: ../data/interim/regulation_events_atomic(~2015).csv
총 155행
